# (Edited Version) Measuring Word Similarity with BERT -- English Language Public Domain Poems

By [The AI for Humanists](https://www.aiforhumanists.com/) Team


How can we measure the similarity of words in a collection of texts? For example, how similar are the words "nature" and "science" in a collection of 16th-20th century English language poems? Do 20th-century poets use the word "science" differently than 16th-century poets? Can we map all the different uses and meanings of the word "nature"?

The short answer is: yes! We can explore all of these questions with BERT, a natural language processing model that has revolutionized the field.

BERT turns words or tokens into vectors — essentially, a list of numbers in a coordinate system (x, y). We can then use the geometric similarity between these resulting vectors as a way to represent varying types of similarity between words.

## In This Notebook
In this Colab notebook, we will specifically analyze a collection of poems scraped from [Public-Domain-Poetry.com](http://public-domain-poetry.com/) with the [DistilBert model](https://huggingface.co/transformers/model_doc/distilbert.html) and the HuggingFace Python library. DistilBert is a smaller — yet still powerful! — version of BERT. By using the rich representations of words that BERT produces, we will then explore the multivalent meanings of particular words in context and over time.

We hope this notebook will help illustrate how BERT works, how well it works, and how you might use BERT to explore the similarity of words in a collection of texts. It is surprising, for example, that BERT works as well as it does, without any fine-tuning, on poems that were published hundreds of years before the text data it was trained on (Wikiepdia pages and self-published novels).

But we also hope that these results will expose some of the limitations and challenges of BERT. We have to disregard poetic line breaks, for example, and we see that BERT has trouble with antiquated words like "thine," which don't show up in its contemporary vocabulary.

The plot above displays a preview of our later results. This is what we're working toward!

You can hover over each point to see the instance of each word in context. If you press `Shift` and click on a point, you will be taken to the original poem on Public-Domain-Poetry.com. Try it out!

## **Import necessary Python libraries and modules**

Ok enough introduction! Let's get started.

To use the HuggingFace [`transformers` Python library](https://huggingface.co/transformers/installation.html), we first need to install it with `pip`.

In [2]:
# !pip install transformers

Then we will import the DistilBertModel and DistilBertTokenizerFast from the Hugging Face `transformers` library. We will also import a handful of other Python libraries and modules.

In [3]:
# For BERT
from transformers import DistilBertTokenizerFast, DistilBertModel

# For data manipulation and analysis
import pandas as pd
pd.options.display.max_colwidth = 200
import numpy as np
from sklearn.decomposition import PCA

# For interactive data visualization
import altair as alt

## **Load text dataset**

Our dataset contains around ~30 thousand poems scraped from  http://public-domain-poetry.com/ (*the link doesn't work now*). This website hosts a curated collection of poems that have fallen out of copyright, which makes them easier for us to share on the web.
You can find the data in our [GitHub repository](https://github.com/melaniewalsh/BERT-4-Humanists/blob/main/data/public-domain-poetry.csv).

We don't have granular date information about when each poem was published, but we do know the birth dates of most of our authors, which we've used to loosely categorize the poems by time period. The poems in our data range from the Middle Ages to the 20th Century, but most come from the 19th Century. The data features both well-known authors — William Wordsworth, Emily Elizabeth Dickinson, Paul Laurence Dunbar, Walt Whitman, Shakespeare — as well as less well-known authors.

Below we will use the Python library `pandas` to read in our CSV file of poems. It is convenient (especially for Colab notebooks) that `pandas` allows you to read in files directly from the web.

To be clear, however, knowledge of `pandas` is not necessary to use BERT. This is simply how we've chosen to load our data. All you really need  is a list of texts (poems, passages, etc.). You can create this list however you are most comfortable.

In [4]:
url = "https://raw.githubusercontent.com/melaniewalsh/BERT-4-Humanists/main/data/public-domain-poetry.csv"

poetry_df = pd.read_csv(url, encoding='utf-8')
# Show 5 random rows
poetry_df.sample(5)

,author,title,text,lifespan,birth_year,death_year,link,period
15557,Madison Julius Cawein,"Beautiful-Bosomed, O Night","I.\n\r\nBeautiful-bosomed, O Night, in thy noon\r\nMove with majesty onward! soaring, as lightly\r\nAs a singer may soar the notes of an exquisite tune,\r\nThe stars and the moon\r\nThrough the cl...",23 March 1865 - 8 December 1914,1865.0,1914.0,http://public-domain-poetry.com/madison-julius-cawein/beautiful-bosomed-o-night-11874,19th Century
24093,Thomas Gent,"1827; Or, The Poet's Last Poem","Ye Bards in all your thousand dens,\r\nGreat souls with fewer pence than pens,\r\nSublime adorers of Apollo,\r\nWith folios full, and purses hollow;\r\nWhose very souls with rapture glisten,\r\nWh...",1780 - ?,1780.0,NaN,http://public-domain-poetry.com/thomas-gent/1827-or-the-poets-last-poem-16242,19th Century
12545,John Greenleaf Whittier,My Dream,"In my dream, methought I trod,\r\nYesternight, a mountain road;\r\nNarrow as Al Sirat's span,\r\nHigh as eagle's flight, it ran.\n\r\nOverhead, a roof of cloud\r\nWith its weight of thunder bowed;...","December 17, 1807-September 7, 1892",1807.0,1892.0,http://public-domain-poetry.com/john-greenleaf-whittier/my-dream-5979,19th Century
13407,John Keats,Answer To A Sonnet By J.H.Reynolds,"Dark eyes are dearer far\r\nThan those that mock the hyacinthine bell.\n\r\nBlue! 'Tis the life of heaven, the domain\r\nOf Cynthia, the wide palace of the sun,\r\nThe tent of Hesperus, and all hi...",31 October 1795-23 February 1821,1795.0,1821.0,http://public-domain-poetry.com/john-keats/answer-to-a-sonnet-by-jhreynolds-6252,19th Century
13123,John Hartley,All Tawk,Some tawk becoss they think they're born\r\nWi' sich a lot o' wit;\r\nSome seem to tawk to let fowk know\r\nThey're born withaat a bit.\r\nSome tawk i' hooaps 'at what they say\r\nMay help ther fe...,1839-1915,1839.0,1915.0,http://public-domain-poetry.com/john-hartley/all-tawk-17954,19th Century


Let's check to see how many poems are in this dataset:

In [5]:
len(poetry_df)

31080

#### **In-class Practice: Exploratory data analysis**
1. Let's check to see which authors show up the most in this dataset (e.g., top 20) to get a sense of its contours:

In [6]:
authors = poetry_df["author"]
author_counts = {}

for author in authors:
    if author in author_counts:
        author_counts[author] += 1
    else:
        author_counts[author] = 1

author_counts = sorted(author_counts.items(), key=lambda item: item[1], reverse=True)
author_counts

[('Robert Herrick', 1464),
 ('Madison Julius Cawein', 1345),
 ('William Wordsworth', 963),
 ('Thomas Moore', 853),
 ('Thomas Hardy', 655),
 ('Rudyard Kipling', 638),
 ('Robert Burns', 499),
 ('John Greenleaf Whittier', 481),
 ('Algernon Charles Swinburne', 461),
 ('Emily Elizabeth Dickinson', 447),
 ('Paul Laurence Dunbar', 417),
 ('John Clare', 382),
 ('William Butler Yeats', 378),
 ('Francesco Petrarca (Petrarch)', 375),
 ('Paul Cameron Brown', 341),
 ('Walt Whitman', 338),
 ('Edgar Lee Masters', 331),
 ('Percy Bysshe Shelley', 330),
 ('Oliver Wendell Holmes', 329),
 ('Walter De La Mare', 329),
 ('Alfred Lord Tennyson', 325),
 ('William Cowper', 308),
 ('Henry Wadsworth Longfellow', 295),
 ('John Hartley', 294),
 ('Banjo Paterson (Andrew Barton)', 283),
 ('Edward Lear', 264),
 ('Friedrich Schiller', 256),
 ('Christina Georgina Rossetti', 241),
 ('Charles Baudelaire', 234),
 ('William Lisle Bowles', 232),
 ('Ralph Waldo Emerson', 226),
 ('Michael Drayton', 225),
 ('Henry Kendall', 214

2. Let's check to see what time periods show up the most in this dataset to get a sense of its contours:

In [7]:
periods = poetry_df['period']
periods = periods.dropna()
time_periods = {}

for period in periods:
    if period in time_periods:
        time_periods[period] += 1
    else:
        time_periods[period] = 1

time_periods = sorted(time_periods.items(), key=lambda item: item[1], reverse=True)
time_periods

[('19th Century', 19025),
 ('20th Century', 4190),
 ('18th Century', 3132),
 ('16th-17th Centuries (Early Modern)', 2786),
 ('12th-15th Centuries (Middle English)', 484)]

## **Sample text dataset**

Though we wish we could analyze all the poems in this data, Colab tends to crash if we try to use more than 4-5,000 poems —  even with DistilBert, the smaller version of BERT. This is an important limitation to keep in mind. If you'd like to use more text data, you might consider upgrading to a paid version of Colab (with more memory or GPUs) or using a compute cluster.

To reduce the number of poems, we will take a random sample of 1,000 poems from four different time periods: the 20th Century, 19th Century, 18th Century, and the Early Modern period.

In [8]:
# Filter the DataFrame for only a given time period, then randomly sample 1000 rows
nineteenth_sample = poetry_df[poetry_df['period'] == '19th Century'].sample(1000)
twentieth_sample = poetry_df[poetry_df['period'] == '20th Century'].sample(1000)
eighteenth_sample = poetry_df[poetry_df['period'] == '18th Century'].sample(1000)
sixteenth_sample = poetry_df[poetry_df['period'] == '16th-17th Centuries (Early Modern)'].sample(1000)

In [9]:
# Merge these random samples into a new DataFrame
poetry_df = pd.concat([sixteenth_sample, eighteenth_sample, twentieth_sample, nineteenth_sample])

In [10]:
poetry_df['period'].value_counts()

period
16th-17th Centuries (Early Modern)    1000
18th Century                          1000
20th Century                          1000
19th Century                          1000
Name: count, dtype: int64

Finally, let's make a list of poems from our Pandas DataFrame.

In [11]:
poetry_texts = poetry_df['text'].tolist()

Let's examine a poem in our dataset:

In [12]:
len(poetry_texts)

4000

In [13]:
print(poetry_texts[0])

Nothing can be more loathsome than to see
Power conjoin'd with Nature's cruelty.


===================



## **Encode/tokenize text data for BERT**

Next we need to transform our poems into a format that BERT (via Huggingface) will understand. This is called *encoding* or *tokenizing* the data.

We will tokenize the poems with the `tokenizer()` from HuggingFace's `DistilBertTokenizerFast`. Here's what the `tokenizer()` will do:

1. Truncate the texts if they're more than 512 tokens or pad them if they're fewer than 512 tokens. If a word is not in BERT's vocabulary, it will be broken up into smaller "word pieces," demarcated by a `##`.

2. Add in special tokens to help BERT:
    - [CLS] — Start token of every document
    - [SEP] — Separator between each sentence
    - [PAD] — Padding at the end of the document as many times as necessary, up to 512 tokens
    - &#35;&#35; — Start of a "word piece"

Here we will load `DistilBertTokenizerFast` from HuggingFace library, which will help us transform and encode the texts so they can be used with BERT.

In [14]:
from transformers import DistilBertTokenizerFast

In [15]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

The `tokenizer()` will break word tokens into word pieces, truncate to 512 tokens, and add padding and special BERT tokens.

In [16]:
tokenized_poems = tokenizer(poetry_texts, truncation=True, padding=True, return_tensors="pt")

Let's examine the first tokenized poem. We can see that the special BERT tokens have been inserted where necessary.

In [17]:
' '.join(tokenized_poems[0].tokens)

"[CLS] nothing can be more lo ##ath ##some than to see power con ##jo ##in ' d with nature ' s cruelty . [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [

<br><br>

## **Load pre-trained BERT model**

Here we will load a pre-trained BERT model. To speed things up we will use a GPU, but using GPU involves a few extra steps.
The command `.to("cuda")` moves data from regular memory to the GPU's memory.




In [18]:
from transformers import DistilBertModel

In [19]:
model = DistilBertModel.from_pretrained('distilbert-base-uncased').to("cuda")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **Get BERT word embeddings for each document in a collection**

To get word embeddings for all the words in our collection, we will use a `for` loop.

For each poem in our list `poetry_texts`, we will tokenize the poem, and we will extract the vocabulary word ID for each word/token in the poem (to use for later reference). Then we will run the tokenized poem through the BERT model and extract the vectors for each word/token in the poem.

We thus create two big lists for all the poems in our collection — `doc_word_ids` and `doc_word_vectors`.

In [20]:
# List of vocabulary word IDs for all the words in each document (aka each poem)
doc_word_ids = []
# List of word vectors for all the words in each document (aka each poem)
doc_word_vectors = []

# Below we will slice our poem to ignore the first (0th) and last (-1) special BERT tokens
start_of_words = 1
end_of_words = -1

# Below we will index the 0th or first document, which will be the only document, since we're analzying one poem at a time
first_document = 0

for i, poem in enumerate(poetry_texts):

    # Here we tokenize each poem with the DistilBERT Tokenizer
    inputs = tokenizer(poem, return_tensors="pt", truncation=True, padding=True)

    # Here we extract the vocabulary word ids for all the words in the poem (the first or 0th document, since we only have one document)
    # We ignore the first and last special BERT tokens
    # We also convert from a Pytorch tensor to a numpy array
    doc_word_ids.append(inputs.input_ids[first_document].numpy()[start_of_words:end_of_words])

    # Here we send the tokenized poems to the GPU
    # The model is already on the GPU, but this poem isn't, so we send it to the GPU
    inputs.to("cuda")
    # Here we run the tokenized poem through the DistilBERT model
    outputs = model(**inputs)

    # We take every element from the first or 0th document, from the 2nd to the 2nd to last position
    # Grabbing the last layer is one way of getting token vectors. There are different ways to get vectors with different pros and cons
    doc_word_vectors.append(outputs.last_hidden_state[first_document,start_of_words:end_of_words,:].detach().cpu().numpy())


Confirm that we have the same number of documents for both the tokens and the vectors:

In [21]:
len(doc_word_ids), len(doc_word_vectors)

(4000, 4000)

In [22]:
doc_word_ids[0], doc_word_vectors[0]

(array([ 2498,  2064,  2022,  2062,  8840,  8988, 14045,  2084,  2000,
         2156,  2373,  9530,  5558,  2378,  1005,  1040,  2007,  3267,
         1005,  1055, 18186,  1012], dtype=int64),
 array([[-0.04058561, -0.01384196, -0.41690552, ...,  0.09674229,
          0.62953466,  0.06135523],
        [ 0.22985075,  0.3657562 , -0.2876274 , ..., -0.1577751 ,
          0.03393744,  0.34653556],
        [ 0.21734008,  0.04313137, -0.42023882, ..., -0.08093612,
         -0.05767591,  0.26987246],
        ...,
        [ 0.3843508 ,  0.09319965, -0.56893474, ..., -0.33802563,
          0.00715934,  0.37065113],
        [ 0.6680031 ,  0.27143028, -0.32429633, ..., -0.1073723 ,
         -0.04309234, -0.11918481],
        [ 0.790368  ,  0.07736056, -0.44672772, ..., -0.04817326,
         -0.44019973, -0.40373713]], dtype=float32))

## **Concatenate all word IDs/vectors for all documents**

Each element of these lists contains all the tokens/vectors for one document. But we want to concatenate them into two giant collections.

In [23]:
all_word_ids = np.concatenate(doc_word_ids)
all_word_vectors = np.concatenate(doc_word_vectors, axis=0)

We want to make comparisons between vectors quickly. One common option is *cosine similarity*, which measures the angle between vectors but ignores their length. We can speed this computation up by setting all the poem vectors to have length 1.0.

In [24]:
# Calculating the length of each vector (Pythagorean theorem)
row_norms = np.sqrt(np.sum(all_word_vectors ** 2, axis=1))
# Dividing every vector by its length
all_word_vectors /= row_norms[:,np.newaxis]

## **Find all word positions in a collection**

We can use the array `all_word_ids` to find all the places, or *positions*, in the collection where a word appears.

We can find a word's vocab ID in BERT with `tokenizer.vocab` and then check to see where/how many times this ID occurs in `all_word_ids`.

In [25]:
def get_word_positions(words):

  """This function accepts a list of words, rather than a single word"""

  # Get word/vocabulary ID from BERT for each word
  word_ids = [tokenizer.vocab[word] for word in words]

  # Find all the positions where the words occur in the collection
  word_positions = np.where(np.isin(all_word_ids, word_ids))[0]

  return word_positions

Here we'll check to see all the places where the word "bank" appears in the collection.

In [26]:
get_word_positions(["bank"])

array([ 30707,  62394,  71959,  75094,  93172,  93616, 173631, 184498,
       196492, 196629, 237880, 243507, 251480, 267254, 270552, 294294,
       326390, 332997, 347405, 347728, 373113, 382068, 414620, 423255,
       440662, 445271, 458543, 483896, 538514, 555857, 559517, 605622,
       636843, 685848, 783947, 814755, 836688, 843710, 855874, 856605,
       857665, 882869, 884999, 895041, 928699], dtype=int64)

In [27]:
word_positions = get_word_positions(["bank"])

## **Find word from word position**

Nice! Now we know all the positions where the word "bank" appears in the collection. But it would be more helpful to know the actual words that appear in context around it. To find these context words, we have to convert position IDs back into words.

In [28]:
# Here we create an array so that we can go backwards from numeric token IDs to words
word_lookup = np.empty(tokenizer.vocab_size, dtype="O")

for word, index in tokenizer.vocab.items():
    word_lookup[index] = word

Now we can use `word_lookup` to find a word based on its position in the collection.

In [29]:
word_positions = get_word_positions(["bank"])

for word_position in word_positions:
  print(word_position, word_lookup[all_word_ids[word_position]])

30707 bank
62394 bank
71959 bank
75094 bank
93172 bank
93616 bank
173631 bank
184498 bank
196492 bank
196629 bank
237880 bank
243507 bank
251480 bank
267254 bank
270552 bank
294294 bank
326390 bank
332997 bank
347405 bank
347728 bank
373113 bank
382068 bank
414620 bank
423255 bank
440662 bank
445271 bank
458543 bank
483896 bank
538514 bank
555857 bank
559517 bank
605622 bank
636843 bank
685848 bank
783947 bank
814755 bank
836688 bank
843710 bank
855874 bank
856605 bank
857665 bank
882869 bank
884999 bank
895041 bank
928699 bank


We can also look for the 3 words that come before "bank" and the 3 words that come after it.

In [30]:
word_positions = get_word_positions(["bank"])

for word_position in word_positions:

  # Slice 3 words before "bank"
  start_pos = word_position - 3
  # Slice 3 words after "bank"
  end_pos = word_position + 4

  context_words = word_lookup[all_word_ids[start_pos:end_pos]]
  # Join the words together
  context_words = ' '.join(context_words)
  print(word_position, context_words)

30707 then on the bank of jordan ,
62394 pride above thy bank ##es . "
71959 pri ##m ##rose bank did cross her
75094 a sunni ##e bank ##e outstretched lay
93172 ##tty * * bank , the which
93616 sober stream , bank ' d all
173631 , the blushing bank is all my
184498 lily by the bank , the pri
196492 on a summer bank , to sing
196629 bends on the bank , amid the
237880 ##s fra ##e bank to bra ##e
243507 the gravel ##ly bank thrown up by
251480 to the warm bank below , yellow
267254 , on some bank beside a river
270552 i . now bank an ' bra
294294 stood upon a bank , above her
326390 " see this bank - note -
332997 river ' s bank , in fable
347405 sunshine on the bank : no tear
347728 residence at allan bank . the long
373113 ##us on the bank , in vain
382068 ##no ' s bank , and on
414620 gently the green bank it la ##ves
423255 hedge , and bank , and stil
440662 george ' s bank when winds were
445271 a moss ##y bank ; where violet
458543 , though no bank would back '
483896 longing to re

#### **In-class Question:**
In the above output, why we have many tokens like "##th", "##lving"?

**Answer:**

Our model is breaking the words down so that it can handle words more efficiently. ## is a special marker

Let's make some functions that will help us get the context words around a certain word position for whatever size window (certain number of words before and after) that we want.

The first function `get_context()` will simply return the tokens without cleaning them, and the second function `get_context_clean()` will return the tokens in a more readable fashion.

In [31]:
def get_context(word_id, window_size=10):

  """Simply get the tokens that occur before and after word position"""

  start_pos = max(0, word_id - window_size) # The token where we will start the context view
  end_pos = min(word_id + window_size + 1, len(all_word_ids)) # The token where we will end the context view

  # Make a list called tokens and use word_lookup to get the words for given token IDs from starting position up to the keyword
  tokens = [word_lookup[word] for word in all_word_ids[start_pos:end_pos] ]

  context_words = " ".join(tokens)

  return context_words

In [32]:
import re

def get_context_clean(word_id, window_size=10):

  """Get the tokens that occur before and after word position AND make them more readable"""

  keyword = word_lookup[all_word_ids[word_id]]
  start_pos = max(0, word_id - window_size) # The token where we will start the context view
  end_pos = min(word_id + window_size + 1, len(all_word_ids)) # The token where we will end the context view

  # Make a list called tokens and use word_lookup to get the words for given token IDs from starting position up to the keyword
  tokens = [word_lookup[word] for word in all_word_ids[start_pos:end_pos] ]

  # Make wordpieces slightly more readable
  # This is probably not the most efficient way to clean and correct for weird spacing
  context_words = " ".join(tokens)
  context_words = re.sub(r'\s+([##])', r'\1', context_words)
  context_words = re.sub(r'##', r'', context_words)
  context_words = re.sub('\s+\'s', '\'s', context_words)
  context_words = re.sub('\s+\'d', '\'d', context_words)
  context_words = re.sub('\s\'er', '\'er', context_words)
  context_words = re.sub(r'\s+([-,:?.!;])', r'\1', context_words)
  context_words = re.sub(r'([-\'"])\s+', r'\1', context_words)
  context_words = re.sub('\s+\'s', '\'s', context_words)
  context_words = re.sub('\s+\'d', '\'d', context_words)

  # Bold the keyword by putting asterisks around it
  if keyword in context_words:
    context_words = re.sub(f"\\b{keyword}\\b", f"**{keyword}**", context_words)
    context_words = re.sub(f"\\b({keyword}[esdtrlying]+)\\b", fr"**\1**", context_words)

  return context_words

<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
<>:24: SyntaxWarning: invalid escape sequence '\s'
<>:25: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
<>:24: SyntaxWarning: invalid escape sequence '\s'
<>:25: SyntaxWarning: invalid escape sequence '\s'
C:\Users\leoxi\AppData\Local\Temp\ipykernel_29284\3190916756.py:19: SyntaxWarning: invalid escape sequence '\s'
  context_words = re.sub('\s+\'s', '\'s', context_words)
C:\Users\leoxi\AppData\Local\Temp\ipykernel_29284\3190916756.py:20: SyntaxWarning: invalid escape sequence '\s'
  context_words = re.sub('\s+\'d', '\'d', context_words)
C:\Users\leoxi\AppData\Local\Temp\ipykernel_29284\3190916756.py:21: SyntaxWarning: invalid escape sequence '\s'
  context_words = re.sub('\s\'er', '\'er

To visualize the search keyword even more easily, we're going to import a couple of Python modules that will allow us to output text with bolded words and other styling. Here we will make a function `print_md()` that will allow us to print with Markdown styling.

In [33]:
from IPython.display import Markdown, display

def print_md(string):
    display(Markdown(string))

In [34]:
word_positions = get_word_positions(["bank"])

for word_position in word_positions:

  print_md(f"<br> {word_position}:  {get_context_clean(word_position)} <br>")

<br> 30707:  but return'd in vain. then on the **bank** of jordan, by a creek: where winds with <br>

<br> 62394:  up, rise, and swell with pride above thy **bankes**. "now is not every tide. tame <br>

<br> 71959:  , and running therewithal a primrose **bank** did cross her way, and gave my love a <br>

<br> 75094:  shore of muddie nile, upon a sunnie **banke** outstretched lay, in monstrous length, a might <br>

<br> 93172:  silver streaming themmes; whose rutty * * **bank**, the which his river hemmes, was pay <br>

...

<br> 857665:  wearied and oppressed, upon a flowery **bank** i laid me down to rest; beneath my feet <br>

<br> 882869:  e, she sat her downe upon a green **bank**, and her true love came riding bye. she <br>

<br> 884999:  s mounds are strown; cave in the **bank** where the sly stream steals; aloe that stab <br>

<br> 895041:  bird, while we follow, like thee, by **bank**, shoal, and quicksand, the <br>

<br> 928699:  eau, gay with loves, lie there beneath a **bank** of eglantine, that heaps a <br>

Here we make a list of all the context views for our keyword.

In [35]:
word_positions = get_word_positions(["bank"])

keyword_contexts = []
keyword_contexts_tokens = []

for position in word_positions:

  keyword_contexts.append(get_context_clean(position))
  keyword_contexts_tokens.append(get_context(position))

=========================

## **Get word vectors and reduce them with PCA**

Finally, we don't just want to *read* all the instances of "bank" in the collection, we want to *measure* the similarity of all the instances of "bank."

To measure similarity between all the instances of "bank," we will take the vectors for each instance and then use PCA to reduce each 768-dimensionsal vector to the 2 dimensions that capture the most variation.

In [36]:
from sklearn.decomposition import PCA

word_positions = get_word_positions(["bank"])

pca = PCA(n_components=2)

pca.fit(all_word_vectors[word_positions,:].T)

PCA(n_components=2)

Then, for convenience, we will put these PCA results into a Pandas DataFrame, which will use to generate an interactive plot.

In [37]:
df = pd.DataFrame({"x": pca.components_[0,:], "y": pca.components_[1,:],
                   "context": keyword_contexts, "tokens": keyword_contexts_tokens})
df.head()

,x,y,context,tokens
0,0.148578,0.242880,"but return'd in vain. then on the **bank** of jordan, by a creek: where winds with","but return ' d in vain . then on the bank of jordan , by a creek : where winds with"
1,0.139883,-0.066744,"up, rise, and swell with pride above thy **bankes**. ""now is not every tide. tame","up , rise , and swell with pride above thy bank ##es . "" now is not every tide . tame"
2,0.146224,-0.264926,", and running therewithal a primrose **bank** did cross her way, and gave my love a",", and running there ##with ##al a pri ##m ##rose bank did cross her way , and gave my love a"
3,0.142409,-0.088336,"shore of muddie nile, upon a sunnie **banke** outstretched lay, in monstrous length, a might","shore of mud ##die nile , upon a sunni ##e bank ##e outstretched lay , in monstrous length , a might"
4,0.150535,-0.053273,"silver streaming themmes; whose rutty * * **bank**, the which his river hemmes, was pay","silver streaming them ##mes ; whose ru ##tty * * bank , the which his river hem ##mes , was pay"


## **Match context with original text and metadata**

It's helpful (and fun!) to know where each instance of a word actually comes from — which poem, which poet, which time period, which Public-Domain-Poetry.com web page. The easiest method we've found for matching a bit of context with its original poem and metdata is to 1) add a tokenized version of each poem to our original Pandas Dataframe 2) check to see if the context shows up in a poem 3) and if so, grab the original poem and metadata.

In [38]:
# Tokenize all the poems
tokenized_poems = tokenizer(poetry_texts, truncation=True, padding=True, return_tensors="pt")

# Get a list of all the tokens for each poem
all_tokenized_poems = []
for i in range(len(tokenized_poems['input_ids'])):
  all_tokenized_poems.append(' '.join(tokenized_poems[i].tokens))

# Add them to the original DataFrame
poetry_df['tokens'] = all_tokenized_poems

In [39]:
poetry_df.head(2)

,author,title,text,lifespan,birth_year,death_year,link,period,tokens
20801,Robert Herrick,Cruelty Base In Commanders,Nothing can be more loathsome than to see\r\nPower conjoin'd with Nature's cruelty.,"Baptized - August 24, 1591- October 1674",1591.0,1674.0,http://public-domain-poetry.com/robert-herrick/cruelty-base-in-commanders-18958,16th-17th Centuries (Early Modern),[CLS] nothing can be more lo ##ath ##some than to see power con ##jo ##in ' d with nature ' s cruelty . [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [P...
20608,Robert Herrick,Calling And Correcting,"God is not only merciful to call\r\nMen to repent, but when He strikes withal.","Baptized - August 24, 1591- October 1674",1591.0,1674.0,http://public-domain-poetry.com/robert-herrick/calling-and-correcting-19502,16th-17th Centuries (Early Modern),"[CLS] god is not only mer ##ciful to call men to rep ##ent , but when he strikes with ##al . [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [..."


In [40]:
def find_original_poem(rows):

  """This function checks to see whether the context tokens show up in the original poem,
  and if so, returns metadata about the title, author, period, and URL for that poem"""

  text = rows['tokens'].replace('**', '')
  text = text[55:70]

  if poetry_df['tokens'].str.contains(text, regex=False).any() == True :
    row = poetry_df[poetry_df['tokens'].str.contains(text, regex=False)].values[0]
    title, author, period, link = row[0], row[1], row[7], row[6]
    return author, title, period, link
  else:
    return None, None, None, None

In [41]:
df[['title', 'author', 'period', 'link']] = df.apply(find_original_poem, axis='columns', result_type='expand')

In [42]:
df

,x,y,context,tokens,title,author,period,link
0,0.148578,0.242880,"but return'd in vain. then on the **bank** of jordan, by a creek: where winds with","but return ' d in vain . then on the bank of jordan , by a creek : where winds with",Paradise Regained - The Second Book,John Milton,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/john-milton/paradise-regained-the-second-book-8323
1,0.139883,-0.066744,"up, rise, and swell with pride above thy **bankes**. ""now is not every tide. tame","up , rise , and swell with pride above thy bank ##es . "" now is not every tide . tame",The Speeches Of Gratulations,Ben Jonson,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/ben-jonson/speeches-of-gratulations-2539
2,0.146224,-0.264926,", and running therewithal a primrose **bank** did cross her way, and gave my love a",", and running there ##with ##al a pri ##m ##rose bank did cross her way , and gave my love a",A Song Upon Silvia,Robert Herrick,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/robert-herrick/song-upon-silvia-19163
3,0.142409,-0.088336,"shore of muddie nile, upon a sunnie **banke** outstretched lay, in monstrous length, a might","shore of mud ##die nile , upon a sunni ##e bank ##e outstretched lay , in monstrous length , a might",Visions Of The Worlds Vanitie,Edmund Spenser,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/edmund-spenser/visions-of-the-worlds-vanitie-32234
4,0.150535,-0.053273,"silver streaming themmes; whose rutty * * **bank**, the which his river hemmes, was pay","silver streaming them ##mes ; whose ru ##tty * * bank , the which his river hem ##mes , was pay","Prothalamion: Or, A Spousall Verse",Edmund Spenser,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/edmund-spenser/prothalamion-or-a-spousall-verse-32239
5,0.132393,-0.186152,"to flow, like to a solemn sober stream, **bank**'d all with lilies, and the cream","to flow , like to a solemn sober stream , bank ' d all with lil ##ies , and the cream",The Wassail,Robert Herrick,16th-17th Centuries (Early Modern),http://public-domain-poetry.com/robert-herrick/wassail-2307
6,0.147089,-0.205165,"at th'shepherd's nose, the blushing **bank** is all my care, with hearth so red,","at th ' shepherd ' s nose , the blushing bank is all my care , with hearth so red ,",Blind Man's Buff,William Blake,18th Century,http://public-domain-poetry.com/william-blake/blind-mans-buff-9293
7,0.158698,-0.002662,"t. iii. now blooms the lily by the **bank**, the primrose down the brae;","##t . iii . now blooms the lily by the bank , the pri ##m ##rose down the bra ##e ;","Lament Of Mary, Queen Of Scots, On The Approach Of Spring",Robert Burns,18th Century,http://public-domain-poetry.com/robert-burns/lament-of-mary-queen-of-scots-on-the-approach-of-spring-10010
8,0.151986,0.067832,"green abode, and, seated on a summer **bank**, to sing no earthly music; in a spot","green ab ##ode , and , seated on a summer bank , to sing no earthly music ; in a spot","Cadland,[1] Southampton River",William Lisle Bowles,18th Century,http://public-domain-poetry.com/william-lisle-bowles/cadland-southampton-river-9408
9,0.153606,0.165385,"; from where the pensile birch bends on the **bank**, amid the clustered group of the dark hollie","; from where the pens ##ile birch bends on the bank , amid the clustered group of the dark ho ##llie","Cadland,[1] Southampton River",William Lisle Bowles,18th Century,http://public-domain-poetry.com/william-lisle-bowles/cadland-southampton-river-9408


============================

## **Plot word embeddings**

Lastly, we will plot the words vectors from this DataFrame with the Python data viz library [Altair](https://altair-viz.github.io/gallery/scatter_tooltips.html).

In [43]:
import altair as alt

In [44]:
alt.Chart(df,title="Word Similarity: Bank").mark_circle(size=200).encode(
    alt.X('x',
        scale=alt.Scale(zero=False)
    ), y="y",
    # If you click a point, take you to the URL link
    href="link",
    # The categories that show up in the hover tooltip
    tooltip=['title', 'context', 'author', 'period']
    ).interactive().properties(
    width=500,
    height=500
)

c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.

alt.Chart(...)

## **Plot word embeddings from keywords (all at once!)**

We can put the code from the previous few sections into a single cell and plot the BERT word embeddings for any list of words. Let's look at the words "nature," "religion," "science," and "art."

In [45]:
# List of keywords that you want to compare
keywords = ['nature', 'religion', 'science', 'art']

# How to color the points in the plot. The other option is "period" for time period
color_by = 'word'

# Get all word positions
word_positions = get_word_positions(keywords)

# Get all contexts around the words
keyword_contexts = []
keyword_contexts_tokens = []
words = []

for position in word_positions:
  words.append(word_lookup[all_word_ids[position]])
  keyword_contexts.append(get_context_clean(position))
  keyword_contexts_tokens.append(get_context(position))

# Reduce word vectors with PCA
pca = PCA(n_components=2)
pca.fit(all_word_vectors[word_positions,:].T)

# Make a DataFrame with PCA results
df = pd.DataFrame({"x": pca.components_[0,:], "y": pca.components_[1,:],
                   "context": keyword_contexts, "tokens": keyword_contexts_tokens, "word": words})
# Match original text and metadata
df[['title', 'author', 'period', 'link']] = df.apply(find_original_poem, axis='columns', result_type='expand')

# Rename columns so that the context shows up as the "title" in the tooltip (bigger and bolded)
df = df.rename(columns={'title': 'poem_title', 'context': 'title'})

# Make the plot
alt.Chart(df, title=f"Word Similarity: {', '.join(keywords).title()}").mark_circle(size=200).encode(
    alt.X('x',
        scale=alt.Scale(zero=False)
    ), y="y",
    color= color_by,
    href="link",
    tooltip=['title', 'word', 'poem_title', 'author', 'period']
    ).interactive().properties(
    width=500,
    height=500
)

c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.

alt.Chart(...)

#### **In-class Question:**
Given the above visulization of keywords in different poems, what can you infer? E.g., How similar are the words "nature" and "art" in a collection of 16th-20th century English language poems?

**Answer:** Art and nature sit on differrent y values, but the sit on the similar X values, meaning that the poem represents them in a similar way, but the way they are percieved is affected

#### **In-class Practice:**

Let's examine the words "nature," "religion," "science," and "art" again but this time color the points by their time period.

In [46]:
# List of keywords that you want to compare
keywords = ['nature', 'religion', 'science', 'art']

# How to color the points in the plot. The other option is "period" for time period
color_by = 'period'

# Get all word positions
word_positions = get_word_positions(keywords)

# Get all contexts around the words
keyword_contexts = []
keyword_contexts_tokens = []
words = []

for position in word_positions:
  words.append(word_lookup[all_word_ids[position]])
  keyword_contexts.append(get_context_clean(position))
  keyword_contexts_tokens.append(get_context(position))

# Reduce word vectors with PCA
pca = PCA(n_components=2)
pca.fit(all_word_vectors[word_positions,:].T)

# Make a DataFrame with PCA results
df = pd.DataFrame({"x": pca.components_[0,:], "y": pca.components_[1,:],
                   "context": keyword_contexts, "tokens": keyword_contexts_tokens, "word": words})
# Match original text and metadata
df[['title', 'author', 'period', 'link']] = df.apply(find_original_poem, axis='columns', result_type='expand')

# Rename columns so that the context shows up as the "title" in the tooltip (bigger and bolded)
df = df.rename(columns={'title': 'poem_title', 'context': 'title'})

# Make the plot
alt.Chart(df, title=f"Word Similarity: {', '.join(keywords).title()}").mark_circle(size=200).encode(
    alt.X('x',
        scale=alt.Scale(zero=False)
    ), y="y",
    color= color_by,
    href="link",
    tooltip=['title', 'word', 'poem_title', 'author', 'period']
    ).interactive().properties(
    width=500,
    height=500
)

c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.

alt.Chart(...)

#### **In-class Question:**
Given the above visulization of keywords in different periods, what can you infer? E.g., Do 20th-century poets use the word "science" differently than 16th-century poets?

**Answer:**

#### **In-class Practice:**

Let's compare the words 'head', 'heart', 'eye', 'arm', and 'leg.'

What can you infer from the results?

In [47]:
# List of keywords that you want to compare
keywords = ['head', 'heart', 'eye', 'arm', 'leg']



Try some other words, like "ring"

In [54]:
# List of keywords that you want to compare
keywords = ['ring']

# How to color the points in the plot. The other option is "period" for time period
color_by = 'word'

# Get all word positions
word_positions = get_word_positions(keywords)

# Get all contexts around the words
keyword_contexts = []
keyword_contexts_tokens = []
words = []

for position in word_positions:
  words.append(word_lookup[all_word_ids[position]])
  keyword_contexts.append(get_context_clean(position))
  keyword_contexts_tokens.append(get_context(position))

# Reduce word vectors with PCA
pca = PCA(n_components=2)
pca.fit(all_word_vectors[word_positions,:].T)

# Make a DataFrame with PCA results
df = pd.DataFrame({"x": pca.components_[0,:], "y": pca.components_[1,:],
                   "context": keyword_contexts, "tokens": keyword_contexts_tokens, "word": words})
# Match original text and metadata
df[['title', 'author', 'period', 'link']] = df.apply(find_original_poem, axis='columns', result_type='expand')

# Rename columns so that the context shows up as the "title" in the tooltip (bigger and bolded)
df = df.rename(columns={'title': 'poem_title', 'context': 'title'})

# Make the plot
alt.Chart(df, title=f"Word Similarity: {', '.join(keywords).title()}").mark_circle(size=200).encode(
    alt.X('x',
        scale=alt.Scale(zero=False)
    ), y="y",
    color= color_by,
    href="link",
    tooltip=['title', 'word', 'poem_title', 'author', 'period']
    ).interactive().properties(
    width=500,
    height=500
)

c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
c:\Users\leoxi\anaconda3\Lib\site-packages\altair\utils\core.

alt.Chart(...)

Write to CSV

In [49]:
df.to_csv('bert-word-ring.csv', index=False, encoding='utf-8')

==============================

## **Find word similarity from a specific word position**

We can also search *all* of the vectors for words similar to a query word.

In [50]:
def get_nearest(query_vector, n=100):
  cosines = all_word_vectors.dot(query_vector)
  ordering = np.flip(np.argsort(cosines))
  return ordering[:n]

To do so, we need to find the specific word position of our desired search keyword.

In [51]:
word_positions = get_word_positions(['bank'])

for word_position in word_positions:

  print_md(f"<br> {word_position}: {get_context_clean(word_position)} <br>")

<br> 30707: but return'd in vain. then on the **bank** of jordan, by a creek: where winds with <br>

<br> 62394: up, rise, and swell with pride above thy **bankes**. "now is not every tide. tame <br>

<br> 71959: , and running therewithal a primrose **bank** did cross her way, and gave my love a <br>

<br> 75094: shore of muddie nile, upon a sunnie **banke** outstretched lay, in monstrous length, a might <br>

<br> 93172: silver streaming themmes; whose rutty * * **bank**, the which his river hemmes, was pay <br>

...

<br> 857665: wearied and oppressed, upon a flowery **bank** i laid me down to rest; beneath my feet <br>

<br> 882869: e, she sat her downe upon a green **bank**, and her true love came riding bye. she <br>

<br> 884999: s mounds are strown; cave in the **bank** where the sly stream steals; aloe that stab <br>

<br> 895041: bird, while we follow, like thee, by **bank**, shoal, and quicksand, the <br>

<br> 928699: eau, gay with loves, lie there beneath a **bank** of eglantine, that heaps a <br>

> 186552: , and in our mossy valleys, on some bank beside a river clear, throw thy silk draperies

In [52]:
keyword_position = 186552

In [53]:
contexts = [get_context_clean(token_id) for token_id in get_nearest(all_word_vectors[keyword_position,:])]

for context in contexts:
  print_md(context)

ert her genial sway, and taught the peaceful **world** her power to obey; content, a female of

as his inferior flame the new-enlightened **world** no more should need; he saw a greater sun

rl'd upon th 'affrighted **world**; sword, fire and famine with fell fury met

like a boy's. we have given the **world** our passion we have naught for death but toys

, and hurled upon th 'affrighted **world**? sword, fire, and famine, with fell

...

on each other'shadow forth thee: 'the **world** hath not another ( tho 'all her fair

'd voice proclaim! so shall the listening **world** be told [ 6 ] medus, and cold

to work my will, and at my deeds the **world** shall thrill. my words shall rouse the slumb

through summer air, her lucid fingers hushed the **world** to sleep. o as i stand this latest moon

shoreless hush of region light, through a new **world**, undreamed of, undesired,